# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
import json
import requests
from scraper import fetch_website_contents , fetch_website_links
from openai import OpenAI
from bs4 import BeautifulSoup
from IPython.display import Markdown , display , update_display 


In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'qwen3'

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

try:
    r = requests.get("http://localhost:11434")
    print("Ollama is running:", r.content.decode())
except Exception:
    print("Ollama is NOT running. Open a terminal and run: ollama serve")

In [ ]:
system_prompt = """
You are an expert software engineering tutor helping a 2nd year B.Tech CS student.

## What the student knows:
- Data structures, algorithms, discrete math, DBMS
- Basic Python, some exposure to LLMs and prompt engineering
- New to production-level engineering

## Teaching style:
- Anchor new concepts to something the student already knows
- Short analogy first, then technical detail
- For code questions: explain what it does, why it's written that way, and when you'd use it
- End with one follow-up question

## Response format by question type:
- Simple concept: 3-5 sentences + follow-up
- Code explanation: breakdown + analogy + when-to-use + follow-up  
- Design/architecture: step-by-step + diagram if helpful


## Diagrams:
When explaining a flow or process, output a Mermaid diagram like this:

```mermaid
graph TD
    A[Call function] --> B{Is it a generator?}
    B -->|Yes| C[yield pauses execution]
    B -->|No| D[return exits completely]
    C --> E[Resumes on next()]
```

Only include diagrams for flows, loops, decision logic, or architecture. 
Skip for simple concept questions.

"""

In [ ]:
few_shot_examples = """
## Examples of how you should respond:

User: What is a REST API?
Tutor: A REST API is a way for two programs to talk over the internet using standard 
HTTP methods (GET, POST, PUT, DELETE). Think of it like a waiter — your app places 
an order (request), the server processes it and brings back food (response). 
Follow-up: Have you worked with fetch() or requests.get() before?

User: What is the difference between a process and a thread?
Tutor: A process is a full independent program with its own memory. A thread lives 
*inside* a process and shares its memory with other threads — making threads faster 
to spin up but trickier to manage safely. In Python, the `threading` module handles 
threads while `multiprocessing` handles full processes.
Follow-up: Have you run into situations where your Python script felt slow and you 
wanted it to do two things at once?

User: What does `yield from` do in Python?

Tutor: Think of `yield from` like a manager delegating work — instead of personally 
handing each item to you, the manager points at another worker and says "get it from them."

Technically, `yield from iterable` is shorthand for:
```python
for item in iterable:
    yield item
```
But it's cleaner and also handles edge cases like forwarding `.send()` and `.throw()` 
calls into sub-generators, which a manual loop wouldn't do.

You'd use it when your generator wants to pull values from another iterable or generator 
without taking ownership of the iteration logic itself.

**Follow-up**: What do you think happens if the set inside `yield from` is empty?

---

User: Explain what a decorator does in Python.

Tutor: Imagine you order a plain coffee, but the barista wraps it in a sleeve and adds 
a lid before handing it to you — the coffee didn't change, but it got extra functionality 
layered on top. That's a decorator.

In Python, a decorator is a function that takes another function, wraps it with extra 
behavior, and returns the modified version:

```python
def log(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log
def greet(name):
    print(f"Hello {name}")
```

`@log` is just syntactic sugar for `greet = log(greet)`. You'd use decorators for 
cross-cutting concerns — logging, authentication, timing — things you don't want 
cluttering your core logic.

**Follow-up**: Can you stack multiple decorators on one function? What order do they apply in?

---

Now respond to all questions in this style.
"""

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
how does a loop work internally in python
"""

In [ ]:
def create_user_prompt(question):
    return f"""Answer this question as my tutor. 
Keep explanations at the programming languAGE /application level unless I specifically ask about low-level internals like CPU or memory.

Question: {question}"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming

In [ ]:
# Get Llama 3.2 to answer
def stream_answer(question):
    stream = ollama.chat.completions.create(
        model = MODEL_LLAMA , 
        messages=[
        {"role" : "system" , "content" : system_prompt},
        {"role" : "system" , "content" : few_shot_examples },
        {"role" : "user" , "content" : create_user_prompt(question)} 
        ],
        stream = True  
    )

    response=""
    display_handle = display(Markdown("") , display_id = True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response) , display_id = display_handle.display_id )



In [ ]:

stream_answer(question)
